# Un mini sito dei progetti pubblici di Arezzo

Dati: [OpenCoesione](https://opencoesione.gov.it/) (licenza CC BY 4.0). Premi Play su ogni cella, dall'alto in basso. Non serve installare nulla: Google Colab esegue Python nel browser. Lo scarico richiede circa tre minuti (26 pagine, una richiesta al secondo).

**1. Impostazioni**

In [ ]:
import html, json, os, time, urllib.error, urllib.request
from datetime import date

COMUNE = "arezzo-comune"          # cambia questo valore per un altro Comune
URL = "https://opencoesione.gov.it/it/api/progetti/?format=json&territorio=" + COMUNE + "&page_size=500&page="
os.makedirs("cache", exist_ok=True)

**2. Una funzione che scarica una pagina, con pausa e memoria**

In [ ]:
def scarica(pagina):
    """Scarica una pagina di risultati; se l'hai già scaricata la rilegge dal disco."""
    nome = f"cache/pagina-{pagina}.json"
    if os.path.exists(nome):
        return json.load(open(nome, encoding="utf-8"))
    time.sleep(1)                  # pausa di cortesia: oltre circa una richiesta al secondo il servizio risponde 429
    richiesta = urllib.request.Request(URL + str(pagina), headers={"User-Agent": "mini-sito-didattico/1.0"})
    for tentativo in range(1, 6):
        try:
            with urllib.request.urlopen(richiesta, timeout=90) as risposta:
                dati = json.load(risposta)
            break
        except urllib.error.HTTPError as errore:
            if errore.code != 429 or tentativo == 5:
                raise
            time.sleep(20 * tentativo)     # troppe richieste: aspetta di più e riprova
    json.dump(dati, open(nome, "w", encoding="utf-8"), ensure_ascii=False)
    return dati

**3. Scarica tutte le pagine**

In [ ]:
progetti, pagina = [], 1
while True:
    dati = scarica(pagina)
    progetti += dati["results"]
    if pagina >= dati["last_page"]:
        break
    pagina += 1
print(f"Scaricate {pagina} pagine, {len(progetti)} progetti")

**4. Somma finanziato e pagato, separati**

In [ ]:
euro = lambda testo: float((testo or "0").replace(",", "."))      # "51817680,65" -> 51817680.65
fmt = lambda x: f"{round(x):,}".replace(",", ".") + " €"
finanziato = sum(euro(p["oc_finanz_tot_pub_netto"]) for p in progetti)
pagato = sum(euro(p["tot_pagamenti"]) for p in progetti)
in_corso = sorted((p for p in progetti if p["oc_stato_progetto"] == "In corso"), key=lambda p: -euro(p["oc_finanz_tot_pub_netto"]))
print("Finanziato:", fmt(finanziato), "- Già pagato:", fmt(pagato), "- In corso:", len(in_corso))

**5. Costruisci la pagina**

In [ ]:
righe = "".join(
    f"<tr><td>{html.escape(p['oc_titolo_progetto'])}</td><td>{fmt(euro(p['oc_finanz_tot_pub_netto']))}</td><td>{fmt(euro(p['tot_pagamenti']))}</td></tr>"
    for p in in_corso[:15])

**6. Salvala**

In [ ]:
pagina_html = f"""<!doctype html>
<html lang="it"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Progetti pubblici ad Arezzo</title>
<style>body{{font:18px/1.5 system-ui,sans-serif;max-width:900px;margin:0 auto;padding:16px;color:#1b2330}}
table{{border-collapse:collapse;width:100%}}td,th{{border-bottom:1px solid #d9e0ea;padding:6px;text-align:left}}.n{{font-size:1.6rem;font-weight:700}}</style></head><body>
<p>Fonte: <a href="https://opencoesione.gov.it/">OpenCoesione</a> (licenza CC BY 4.0). Dati scaricati il {date.today().strftime('%d/%m/%Y')}.</p>
<h1>Progetti pubblici ad Arezzo</h1>
<p><span class="n">{fmt(finanziato)}</span> finanziati e <span class="n">{fmt(pagato)}</span> già pagati, su {len(progetti)} progetti.</p>
<h2>I 15 progetti in corso più grandi</h2>
<div style="overflow-x:auto"><table><tr><th>Titolo</th><th>Finanziato</th><th>Già pagato</th></tr>{righe}</table></div>
</body></html>"""

**7. Guarda il risultato** (la pagina costruita sopra, mostrata qui)

In [ ]:
import html
from IPython.display import display
display({"text/html": '<iframe srcdoc="' + html.escape(pagina_html) + '" width="100%" height="520" style="border:1px solid #ccc;background:#fff"></iframe>'}, raw=True)